# ML_U4_C02 — Clustering Jerárquico: Estructura sin Comprometerse con K

📝 **Modalidad: Clase interactiva — sigue junto al profesor.**

**Versión:** 2025-1 | **Modificado:** 2026-05-30

---

## 📋 Mapa de la clase

| Sección | Tema | Tiempo |
|---------|------|--------|
| 1 | Recap K-means + motivación del clustering jerárquico | 10 min |
| 2 | Algoritmo aglomerativo: fusión de clusters | 20 min |
| 3 | Criterios de enlace: single, complete, average, Ward | 30 min |
| 4 | Dendrogramas: leer, cortar e interpretar | 20 min |
| 5 | Aplicación a Iris: comparar con K-means | 15 min |
| 6 | Ejercicio en clase | 10 min |
| 7 | Resumen y cierre de la unidad | 5 min |

---

## 📚 Prerequisitos

### 🔵 Pregrado
- Clase anterior: K-means, inercia, silhouette
- Noción de distancia euclidiana
- Interpretación de dendrogramas a nivel de árbol genealógico

### 🟡 Doctorado
- Todo lo anterior, además:
- K-means como caso límite de EM
- Teoría de grafos mínimos conectados
- Criterio de Ward como minimización de varianza intra-cluster

---

## 🎯 Objetivos de aprendizaje

Al terminar esta clase podrás:
- Describir el algoritmo aglomerativo y aplicarlo con scikit-learn
- Distinguir y elegir entre los criterios de enlace (single, complete, average, Ward)
- Leer un dendrograma y determinar el número de clusters cortando a la altura adecuada
- Comparar clustering jerárquico con K-means y saber cuándo preferir cada uno
- **(Doctorado)** Demostrar que Ward es equivalente a minimizar la inercia total

## ⚙️ Setup (NO MODIFICAR)

In [ ]:
# ── SETUP — NO MODIFICAR ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist, squareform
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Datasets
iris = load_iris()
X_iris, y_iris = iris.data, iris.target
X_iris_sc = StandardScaler().fit_transform(X_iris)

X_blobs, _ = make_blobs(n_samples=150, centers=4, cluster_std=0.7, random_state=RANDOM_STATE)
X_blobs_sc = StandardScaler().fit_transform(X_blobs)

import sklearn, scipy
print(f"✅ Setup completo")
print(f"   numpy {np.__version__} | sklearn {sklearn.__version__} | scipy {scipy.__version__}")
print(f"   Iris: {X_iris.shape} | Blobs: {X_blobs.shape}")

---
## Sección 1 — Recap y Motivación (10 min)

### Limitación clave de K-means

K-means requiere especificar K **antes** de ver los datos. Esto es problemático cuando:
- No tenemos conocimiento del dominio sobre cuántos grupos existen
- Queremos explorar estructuras a **múltiples escalas** (grupos grandes y subgrupos internos)
- El número "correcto" de clusters es ambiguo

### La propuesta del clustering jerárquico

En lugar de un único particionamiento en K clusters, el clustering jerárquico construye
una **jerarquía completa** de particionamientos, representada como un árbol (dendrograma).
Luego podemos "cortar" el árbol a cualquier altura para obtener el K que queramos.

In [ ]:
# ━━━ MOTIVACIÓN: JERARQUÍA EN EL MUNDO REAL ━━━
# Ejemplo: clasificación biológica de Iris
print("Jerarquía biológica (ejemplo):")
print("""  Reino Plantae
    └── Familia Iridaceae
          └── Género Iris
                ├── Iris setosa        (Cluster A)
                ├── Iris versicolor    (Cluster B)
                └── Iris virginica     (Cluster C)
""")
print("💡 Cortar en el nivel 'Género' da 1 cluster. Cortar en el nivel 'Especie' da 3 clusters.")
print("   El dendrograma permite explorar TODOS los niveles de corte con un solo análisis.")

---
## Sección 2 — El Algoritmo Aglomerativo (20 min)

Existen dos enfoques:
- **Aglomerativo** (bottom-up): empieza con N clusters (un punto cada uno) y va fusionando los más similares
- **Divisivo** (top-down): empieza con 1 cluster y va dividiendo

El aglomerativo es por lejos el más utilizado.

### Algoritmo Aglomerativo

1. Inicializar: cada punto es su propio cluster → N clusters
2. Calcular la matriz de distancias entre todos los pares de clusters
3. Fusionar los dos clusters más cercanos según el **criterio de enlace**
4. Actualizar la matriz de distancias
5. Repetir desde 2 hasta que quede 1 solo cluster

**Complejidad:** $O(n^2 \log n)$ en tiempo, $O(n^2)$ en espacio — más lento que K-means para datos grandes.

In [ ]:
# ━━━ DEMOSTRACIÓN PASO A PASO (6 puntos en 1D) ━━━
# Dataset mínimo para seguir el algoritmo manualmente
X_mini = np.array([[1.0], [2.0], [5.0], [6.0], [10.0], [11.0]])
labels_mini = ['A', 'B', 'C', 'D', 'E', 'F']

print("Puntos: A=1, B=2, C=5, D=6, E=10, F=11")
print("\nMatriz de distancias inicial:")
D = squareform(pdist(X_mini, metric='euclidean'))
df_D = pd.DataFrame(D, index=labels_mini, columns=labels_mini)
print(df_D.round(1).to_string())

print("\nPasos del algoritmo aglomerativo (Ward):")
Z = linkage(X_mini, method='ward')
merge_labels = labels_mini.copy()
step_labels = labels_mini.copy()

for i, (i1, i2, dist, n) in enumerate(Z):
    c1 = step_labels[int(i1)]
    c2 = step_labels[int(i2)]
    nuevo = f"({c1}∪{c2})"
    step_labels.append(nuevo)
    print(f"  Paso {i+1}: fusionar {c1} + {c2} → {nuevo}  [dist={dist:.2f}]")

---
## Sección 3 — Criterios de Enlace (30 min)

El **criterio de enlace** define cómo medir la distancia entre dos clusters cuando cada uno
tiene múltiples puntos. La elección del criterio tiene un impacto enorme en la forma del dendrograma.

| Criterio | Distancia entre $C_A$ y $C_B$ |
|----------|-------------------------------|
| **Single linkage** | $\min_{a \in C_A, b \in C_B} d(a,b)$ — el par más cercano |
| **Complete linkage** | $\max_{a \in C_A, b \in C_B} d(a,b)$ — el par más lejano |
| **Average linkage** | $\frac{1}{|C_A||C_B|} \sum_{a,b} d(a,b)$ — promedio de todos los pares |
| **Ward linkage** | Aumento de inercia total al fusionar → minimiza varianza intra-cluster |

In [ ]:
# ━━━ COMPARACIÓN VISUAL DE LOS 4 CRITERIOS DE ENLACE ━━━
methods = ['single', 'complete', 'average', 'ward']
method_names = ['Single Linkage', 'Complete Linkage', 'Average Linkage', 'Ward']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

for col, (method, name) in enumerate(zip(methods, method_names)):
    # Dendrograma
    Z = linkage(X_blobs_sc, method=method)
    ax_dend = axes[0, col]
    dendrogram(Z, ax=ax_dend, truncate_mode='lastp', p=20,
               no_labels=True, color_threshold=0.7*max(Z[:, 2]))
    ax_dend.set_title(f'{name}\nDendrograma', fontsize=10)
    ax_dend.set_ylabel('Distancia' if col == 0 else '')
    ax_dend.set_xlabel('')

    # Clusters (K=4)
    ac = AgglomerativeClustering(n_clusters=4, linkage=method)
    labels = ac.fit_predict(X_blobs_sc)
    sil = silhouette_score(X_blobs_sc, labels)
    ax_clust = axes[1, col]
    ax_clust.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap='tab10',
                     s=25, alpha=0.7, edgecolors='white', linewidths=0.3)
    ax_clust.set_title(f'K=4 | Silhouette={sil:.3f}', fontsize=10)
    ax_clust.set_xticks([]); ax_clust.set_yticks([])

plt.suptitle('Comparación de Criterios de Enlace — Dataset Blobs (K verdadero=4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Ward y Average suelen dar los mejores resultados en datos esféricos")

In [ ]:
# ━━━ EL PROBLEMA DEL ENCADENAMIENTO (CHAINING) EN SINGLE LINKAGE ━━━
from sklearn.datasets import make_moons

X_chain, y_chain = make_moons(n_samples=100, noise=0.1, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
datasets_chain = [
    (AgglomerativeClustering(n_clusters=2, linkage='single'), 'Single (funciona aquí)'),
    (AgglomerativeClustering(n_clusters=2, linkage='complete'), 'Complete (falla)'),
    (AgglomerativeClustering(n_clusters=2, linkage='ward'), 'Ward (falla)'),
]

for ax, (model, name) in zip(axes, datasets_chain):
    labels = model.fit_predict(X_chain)
    ari = adjusted_rand_score(y_chain, labels)
    ax.scatter(X_chain[:, 0], X_chain[:, 1], c=labels, cmap='RdBu',
               s=40, alpha=0.8, edgecolors='white', linewidths=0.3)
    ax.set_title(f'{name}\nARI={ari:.2f}', fontsize=10,
                 color='seagreen' if ari > 0.8 else 'tomato')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Single Linkage para Clusters no Convexos (Lunas)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Single linkage puede seguir formas no esféricas mediante encadenamiento — pero esto puede ser una ventaja o una trampa")

---
## Sección 4 — Dendrogramas: Leer, Cortar e Interpretar (20 min)

El **dendrograma** es la representación visual de la jerarquía de clusters.

- **Eje X**: los puntos (o clusters intermedios)
- **Eje Y**: la distancia a la que ocurrió la fusión
- **Nodos internos**: fusiones entre clusters
- **Altura del nodo**: qué tan disimilares eran los clusters fusionados

### ¿Cómo elegir el número de clusters?

**Regla práctica:** cortar el dendrograma donde hay una **brecha grande** en el eje Y
(un salto en la distancia de fusión). El número de líneas verticales que cruza el corte
horizontal es el número de clusters.

In [ ]:
# ━━━ DENDROGRAMA COMPLETO CON IRIS ━━━
Z_iris = linkage(X_iris_sc, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Dendrograma completo (truncado para legibilidad)
dendrogram(Z_iris, ax=axes[0], truncate_mode='lastp', p=30,
           no_labels=False, color_threshold=4.5,
           above_threshold_color='gray')
axes[0].set_title('Dendrograma Ward — Iris (truncado a 30 nodos)', fontsize=11)
axes[0].set_xlabel('Tamaño del cluster (entre paréntesis)')
axes[0].set_ylabel('Distancia de fusión (Ward)')

# Líneas de corte
for h, label, color in [(4.5, 'K=2 (corte aquí)', 'tomato'),
                          (3.0, 'K=3 (corte aquí)', 'steelblue'),
                          (2.0, 'K=4 (corte aquí)', 'seagreen')]:
    axes[0].axhline(h, color=color, linestyle='--', alpha=0.7, linewidth=1.5,
                    label=label)
axes[0].legend(fontsize=9)

# Histograma de distancias de fusión (para identificar brechas)
fusion_distances = Z_iris[:, 2]
axes[1].barh(range(len(fusion_distances))[-30:],
             fusion_distances[-30:], color='steelblue', alpha=0.7)
axes[1].set_xlabel('Distancia de fusión')
axes[1].set_ylabel('Paso de fusión (los últimos 30)')
axes[1].set_title('Distancias de las últimas fusiones\n(brechas grandes → cortes naturales)', fontsize=11)

# Marcar la brecha más grande
last_30 = fusion_distances[-30:]
diffs = np.diff(last_30)
biggest_gap_idx = np.argmax(diffs)
axes[1].axvline(last_30[biggest_gap_idx + 1], color='tomato', linestyle='--',
                alpha=0.7, label=f'Brecha más grande')
axes[1].legend()

plt.tight_layout()
plt.show()
print("💡 Las brechas grandes en el eje Y del dendrograma sugieren cortes naturales")

In [ ]:
# ━━━ CORTAR EL DENDROGRAMA A DISTINTOS K ━━━
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_iris_2d = pca.fit_transform(X_iris_sc)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Verdad real
for k, name in enumerate(iris.target_names):
    mask = y_iris == k
    axes[0].scatter(X_iris_2d[mask, 0], X_iris_2d[mask, 1],
                    label=name, s=30, alpha=0.8)
axes[0].set_title('Verdad real (3 especies)', fontsize=10)
axes[0].legend(fontsize=8); axes[0].set_xticks([]); axes[0].set_yticks([])

# K=2, 3, 4
for ax, K in zip(axes[1:], [2, 3, 4]):
    labels_k = fcluster(Z_iris, K, criterion='maxclust') - 1
    sil = silhouette_score(X_iris_sc, labels_k)
    ari = adjusted_rand_score(y_iris, labels_k)
    ax.scatter(X_iris_2d[:, 0], X_iris_2d[:, 1], c=labels_k, cmap='tab10',
               s=30, alpha=0.8, edgecolors='white', linewidths=0.3)
    ax.set_title(f'Ward K={K}\nSil={sil:.3f} | ARI={ari:.3f}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Cortes del Dendrograma Ward a K=2,3,4 — Iris', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Sección 5 — Aplicación Completa: Comparar con K-means (15 min)

In [ ]:
# ━━━ COMPARACIÓN SISTEMÁTICA: K-MEANS vs JERÁRQUICO EN IRIS ━━━
from sklearn.metrics import silhouette_score, adjusted_rand_score

K = 3  # número de especies reales

# K-means
km = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
labels_km = km.fit_predict(X_iris_sc)

# Jerárquico (4 variantes)
results = {}
for method in ['single', 'complete', 'average', 'ward']:
    ac = AgglomerativeClustering(n_clusters=K, linkage=method)
    labels = ac.fit_predict(X_iris_sc)
    results[f'Jerárquico ({method})'] = labels

results['K-means'] = labels_km

print(f"Comparación con K={K} — Iris (verdad: {K} especies)\n")
print(f"{'Método':<30} {'Silhouette':>12} {'ARI':>8}")
print("-" * 55)
for name, labels in results.items():
    sil = silhouette_score(X_iris_sc, labels)
    ari = adjusted_rand_score(y_iris, labels)
    best_marker = " ← mejor ARI" if ari == max(adjusted_rand_score(y_iris, l) for l in results.values()) else ""
    print(f"{name:<30} {sil:>12.4f} {ari:>8.4f}{best_marker}")

In [ ]:
# ━━━ HEATMAP DE DISTANCIAS (VISUALIZACIÓN DE LA ESTRUCTURA) ━━━
# Ordenar los puntos según el clustering jerárquico Ward
Z_ward = linkage(X_iris_sc, method='ward')
from scipy.cluster.hierarchy import leaves_list
order = leaves_list(Z_ward)

D_matrix = squareform(pdist(X_iris_sc, metric='euclidean'))
D_ordered = D_matrix[order][:, order]
y_ordered = y_iris[order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap sin ordenar
im0 = axes[0].imshow(D_matrix, cmap='YlOrRd', aspect='auto')
axes[0].set_title('Matriz de distancias (orden original)', fontsize=11)
plt.colorbar(im0, ax=axes[0])

# Heatmap ordenado por clustering
im1 = axes[1].imshow(D_ordered, cmap='YlOrRd', aspect='auto')
axes[1].set_title('Matriz de distancias (ordenada por Ward)\n→ bloques = clusters', fontsize=11)
plt.colorbar(im1, ax=axes[1])

# Líneas para separar los clusters reales
for i in range(1, 3):
    boundary = np.sum(y_ordered < i)
    axes[1].axhline(boundary - 0.5, color='white', linewidth=2)
    axes[1].axvline(boundary - 0.5, color='white', linewidth=2)

plt.suptitle('Heatmap de Distancias — Iris', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("💡 Al ordenar por Ward aparecen bloques diagonales oscuros: los clusters son reales")

---
## Sección 6 — Ejercicio en Clase (10 min)

### Parte A — Sin computador (5 min) 🖊️

Tienes 5 puntos con la siguiente matriz de distancias:

```
    A   B   C   D   E
A   0   1   9   8  10
B   1   0   8   9  11
C   9   8   0   2   7
D   8   9   2   0   6
E  10  11   7   6   0
```

**1.** Aplica **single linkage** manualmente (3 pasos). ¿Qué clusters resultan si K=2?

**2.** ¿Cuál sería el resultado con **complete linkage**? ¿Cambia el resultado para K=2?

**3.** Dibuja esquemáticamente el dendrograma (no necesita ser preciso, solo la topología).

*Escribe tu respuesta aquí antes de ejecutar el código:*

In [ ]:
# ━━━ VERIFICACIÓN PARTE A ━━━
D = np.array([
    [0,  1,  9,  8, 10],
    [1,  0,  8,  9, 11],
    [9,  8,  0,  2,  7],
    [8,  9,  2,  0,  6],
    [10, 11,  7,  6,  0]
], dtype=float)
labels_abcde = ['A', 'B', 'C', 'D', 'E']

from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, dendrogram

D_cond = squareform(D)  # formato condensado

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, method in zip(axes, ['single', 'complete']):
    Z = linkage(D_cond, method=method)
    dendrogram(Z, labels=labels_abcde, ax=ax)
    ax.set_title(f'{method.capitalize()} Linkage\n→ K=2: corte arriba del primer nodo grande', fontsize=10)
    ax.set_ylabel('Distancia')

plt.suptitle('Verificación Ejercicio A: Single vs Complete', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Single linkage K=2:   {A,B} vs {C,D,E}  [primer par: A-B (dist=1)]")
print("Complete linkage K=2: {A,B} vs {C,D,E}  [misma agrupación por la gran brecha]")

---
## Sección 7 — Resumen y Cierre de la Unidad (5 min)

### 📊 Tabla comparativa — K-means vs Clustering Jerárquico

| Aspecto | K-means | Jerárquico (Ward) |
|---------|---------|-------------------|
| **¿Hay que especificar K?** | Sí, antes del análisis | No; se elige después del dendrograma |
| **Complejidad** | $O(n \cdot K \cdot d \cdot T)$ — rápido | $O(n^2 \log n)$ — lento para n grande |
| **Resultado** | Un particionamiento | Jerarquía completa |
| **Reproducibilidad** | Depende de la inicialización | Determinista |
| **Tipo de clusters** | Esféricos | Más flexible (depende del enlace) |
| **¿Cuándo usarlo?** | N grande (>10k), K conocido | N mediano, exploración, visualización |
| **Sensible a outliers** | Sí (media) | Depende del enlace (Ward: sí) |

### 📊 Guía de elección del criterio de enlace

| Situación | Criterio recomendado |
|-----------|---------------------|
| No sabes nada del problema | Ward (default) |
| Clusters alargados o cadenas | Single |
| Quieres clusters compactos y bien separados | Complete |
| Quieres un punto medio | Average (UPGMA) |

### 🔗 Cierre de la Unidad 4 — Clustering

Has completado los dos algoritmos fundamentales de clustering:

| Clase 1 | Clase 2 |
|---------|---------|
| K-means: particionamiento plano | Jerárquico: árbol de clusters |

La **próxima unidad** explora la **reducción de dimensionalidad** (PCA, t-SNE, UMAP) — técnicas complementarias que transforman los datos para visualización y preprocesamiento antes del clustering.

---

### 📚 Bibliografía

#### Pregrado
- Géron, A. (2022). *Hands-On ML* (3ª ed.). Cap. 9: Unsupervised Learning Techniques.
- sklearn: [AgglomerativeClustering](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html)

#### Doctorado / Investigación
- Ward, J. H. (1963). Hierarchical grouping to optimize an objective function. *JASA*, 58(301).
- Lance, G. N., & Williams, W. T. (1967). A general theory of classificatory sorting strategies. *The Computer Journal*.
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning*. Cap. 14.3.
- Murtagh, F., & Legendre, P. (2014). Ward's hierarchical agglomerative clustering method. *Journal of Classification*, 31(3), 274–295.